# Fund Documentation Assistant

Run the setup cell once, then use only the final `ask(...)` cell. The assistant returns the natural-language answer and the assignment-required JSON structure: `answer`, `funds`, and source `document/page/section`.


In [1]:
import os
import sys
import json
import time
import getpass
import importlib
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault(
    "HF_HUB_CACHE",
    str(ROOT / ".cache" / "huggingface" / "hub"),
)

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fund_rag import generator
importlib.reload(generator)

API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    API_KEY = getpass.getpass("Gemini API key: ").strip()

if not API_KEY:
    raise ValueError("Gemini API key is required.")

MODEL = os.environ.get(
    "RAG_GENERATION_MODEL",
    "gemini-3.5-flash",
)

assistant = generator.build_assistant(
    root=ROOT,
    api_key=API_KEY,
    model=MODEL,
    top_k=6,
)

print("Assistant ready.")
print("Retrieval model:", assistant.retriever.model_name)
print("Generator:", assistant.generator.model)


Gemini API key:  ········


Z:\niksan\Environments\Initial Test\OBB\fund-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[retrieval] Loading embedding model: BAAI/bge-m3


Z:\niksan\Environments\Initial Test\OBB\fund-rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in Z:\niksan\Environments\Initial Test\OBB\fund-rag\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is no

[retrieval] Embedding model loaded.
[retrieval] rebuild=False
[retrieval] FAISS path: Z:\niksan\Environments\Initial Test\OBB\fund-rag\artifacts\retrieval_bge_m3\semantic.faiss
[retrieval] FAISS exists: True
[retrieval] metadata exists: True
[retrieval] Cache matches current chunks/model.
[retrieval] Loading existing FAISS index — NO re-embedding.
[retrieval] Loaded existing FAISS index with 1654 vectors.
Assistant ready.
Retrieval model: BAAI/bge-m3
Generator: gemini-3.5-flash


In [4]:
def ask(question: str):
    """Ask a question and print the answer plus the assignment-required JSON output."""
    question = question.strip()
    if not question:
        raise ValueError("Question cannot be empty.")

    result = assistant.answer(question)

    output = {
        "answer": result.answer,
        "funds": result.funds,
        "sources": [
            {
                "document": source.get("document"),
                "page": source.get("page"),
                "section": source.get("section"),
            }
            for source in result.sources
        ],
    }

    print("ANSWER")
    print("=" * 80)
    print(result.answer)

    print("\nJSON")
    print("=" * 80)
    print(json.dumps(
        output,
        ensure_ascii=False,
        indent=2,
    ))

    return output


## Ask

Change only the question below and rerun this cell.


In [5]:
ask("Can I transfer my fund units to another person?")


ANSWER
Yes, fund units can be transferred freely, without restrictions or additional conditions, in accordance with the wishes of their owners and the specific requirements of applicable legislation. The transfer of units (such as through purchase/sale, donation, or inheritance) must be carried out through a licensed investment intermediary acting as a registration agent. All transfers take effect after registration in the account/sub-account of the acquirer at the "Central Depository" AD ("Централен депозитар" АД).

JSON
{
  "answer": "Yes, fund units can be transferred freely, without restrictions or additional conditions, in accordance with the wishes of their owners and the specific requirements of applicable legislation. The transfer of units (such as through purchase/sale, donation, or inheritance) must be carried out through a licensed investment intermediary acting as a registration agent. All transfers take effect after registration in the account/sub-account of the acquirer a

{'answer': 'Yes, fund units can be transferred freely, without restrictions or additional conditions, in accordance with the wishes of their owners and the specific requirements of applicable legislation. The transfer of units (such as through purchase/sale, donation, or inheritance) must be carried out through a licensed investment intermediary acting as a registration agent. All transfers take effect after registration in the account/sub-account of the acquirer at the "Central Depository" AD ("Централен депозитар" АД).',
 'funds': [],
 'sources': [{'document': '11074.pdf',
   'page': 1,
   'section': 'Прехвърляне на дялове'}]}

In [6]:
ask("What is the weather in Sofia?")


ANSWER
I am sorry, but the provided evidence does not contain information about the weather in Sofia.

JSON
{
  "answer": "I am sorry, but the provided evidence does not contain information about the weather in Sofia.",
  "funds": [],
  "sources": []
}


{'answer': 'I am sorry, but the provided evidence does not contain information about the weather in Sofia.',
 'funds': [],
 'sources': []}

In [8]:
ask("What is the investment objective of K&H egészség?")

ANSWER
The K&H egészség sub-fund has two investment objectives (before taxes and charges):
1. To repay at maturity at least 100% of the initial subscription price (specifically 10,000 HUF per share, providing capital protection).
2. To generate a potential capital gain or return, through an investment in swaps, that is contingent on the performance of a basket of 30 shares of companies in the pharmaceutical and healthcare industries.

JSON
{
  "answer": "The K&H egészség sub-fund has two investment objectives (before taxes and charges):\n1. To repay at maturity at least 100% of the initial subscription price (specifically 10,000 HUF per share, providing capital protection).\n2. To generate a potential capital gain or return, through an investment in swaps, that is contingent on the performance of a basket of 30 shares of companies in the pharmaceutical and healthcare industries.",
  "funds": [
    "K&H egészség"
  ],
  "sources": [
    {
      "document": "FU_BF065_EN.pdf",
      "page

{'answer': 'The K&H egészség sub-fund has two investment objectives (before taxes and charges):\n1. To repay at maturity at least 100% of the initial subscription price (specifically 10,000 HUF per share, providing capital protection).\n2. To generate a potential capital gain or return, through an investment in swaps, that is contingent on the performance of a basket of 30 shares of companies in the pharmaceutical and healthcare industries.',
 'funds': ['K&H egészség'],
 'sources': [{'document': 'FU_BF065_EN.pdf',
   'page': 40,
   'section': 'Investment objectives and strategy'},
  {'document': 'FU_BF065_EN.pdf',
   'page': 38,
   'section': "Description of the sub-fund's object"}]}

In [11]:
ask("What is the investment objective of K&H egészség 2?")

ANSWER
The sub-fund K&H egészség 2 has two main investment objectives:

1. The repayment of 100% of the initial subscription price (10,000 HUF per share) at Maturity.
2. A potential capital gain or return based on the performance of a basket of 30 shares of companies active in the healthcare and pharmaceutical sector, which is provided through investments in swaps.

JSON
{
  "answer": "The sub-fund K&H egészség 2 has two main investment objectives:\n\n1. The repayment of 100% of the initial subscription price (10,000 HUF per share) at Maturity.\n2. A potential capital gain or return based on the performance of a basket of 30 shares of companies active in the healthcare and pharmaceutical sector, which is provided through investments in swaps.",
  "funds": [
    "K&H egészség 2"
  ],
  "sources": [
    {
      "document": "FU_BF065_EN.pdf",
      "page": 52,
      "section": "Investment objectives and strategy"
    },
    {
      "document": "FU_BF065_EN.pdf",
      "page": 50,
      "s

{'answer': 'The sub-fund K&H egészség 2 has two main investment objectives:\n\n1. The repayment of 100% of the initial subscription price (10,000 HUF per share) at Maturity.\n2. A potential capital gain or return based on the performance of a basket of 30 shares of companies active in the healthcare and pharmaceutical sector, which is provided through investments in swaps.',
 'funds': ['K&H egészség 2'],
 'sources': [{'document': 'FU_BF065_EN.pdf',
   'page': 52,
   'section': 'Investment objectives and strategy'},
  {'document': 'FU_BF065_EN.pdf',
   'page': 50,
   'section': "Description of the sub-fund's object"}]}